# Import Libraries

In [2]:
# =============================================
# FEATURE ENGINEERING
# EcoType - Forest Cover Type Prediction
# =============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import joblib
import os

%matplotlib inline

print("All libraries imported successfully!")

All libraries imported successfully!


# Load Cleaned Dataset

In [5]:
# Load the cleaned dataset 

df = pd.read_csv('../data/forest_cover_cleaned.csv')

print(f"Dataset loaded! Shape: {df.shape}")
df.head()

Dataset loaded! Shape: (145890, 13)


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Cover_Type,Wilderness_Area,Soil_Type
0,2596.0,51,3,258.0,0,510,221.0,232,148,6279,Aspen,1,29
1,2590.0,56,2,212.0,-6,390,220.0,235,151,6225,Aspen,1,29
2,2804.0,139,9,268.0,65,3180,234.0,238,135,6121,Lodgepole Pine,1,12
3,2785.0,155,18,242.0,117,3090,238.0,238,122,6211,Lodgepole Pine,1,30
4,2595.0,45,2,153.0,-1,391,220.0,234,150,6172,Aspen,1,29


# Define Column Groups

In [8]:
target_col = 'Cover_Type'
categorical_cols = ['Wilderness_Area', 'Soil_Type']
numerical_cols = [col for col in df.columns 
                  if col not in categorical_cols + [target_col]]

print(" Numerical columns:", numerical_cols)
print(" Categorical columns:", categorical_cols)
print(" Target column:", target_col)

 Numerical columns: ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points']
 Categorical columns: ['Wilderness_Area', 'Soil_Type']
 Target column: Cover_Type


# Create New Features (Feature Engineering)

In [11]:
# We create NEW columns from existing ones
# These new features help the ML model learn better patterns

df_eng = df.copy()  # work on a copy

# 1. Average distance to water (combining horizontal + vertical)
df_eng['Mean_Distance_To_Hydrology'] = (
    df_eng['Horizontal_Distance_To_Hydrology'] + 
    df_eng['Vertical_Distance_To_Hydrology']
) / 2

# 2. Average hillshade across all 3 times of day
df_eng['Mean_Hillshade'] = (
    df_eng['Hillshade_9am'] + 
    df_eng['Hillshade_Noon'] + 
    df_eng['Hillshade_3pm']
) / 3

# 3. Elevation to Roadway ratio
df_eng['Elevation_To_Roadway_Ratio'] = (
    df_eng['Elevation'] / 
    (df_eng['Horizontal_Distance_To_Roadways'] + 1)  # +1 avoids division by zero
)

# 4. Difference between morning and afternoon hillshade
# (captures how much sunlight changes during the day)
df_eng['Hillshade_Diff'] = (
    df_eng['Hillshade_9am'] - 
    df_eng['Hillshade_3pm']
)

print(" New features created!")
print(f"Shape before: {df.shape}")
print(f"Shape after:  {df_eng.shape}")
print(f"\nNew columns added:")
print("- Mean_Distance_To_Hydrology")
print("- Mean_Hillshade")
print("- Elevation_To_Roadway_Ratio")
print("- Hillshade_Diff")

 New features created!
Shape before: (145890, 13)
Shape after:  (145890, 17)

New columns added:
- Mean_Distance_To_Hydrology
- Mean_Hillshade
- Elevation_To_Roadway_Ratio
- Hillshade_Diff


#  Encode Wilderness_Area (Label Encoding)

In [14]:
# ML models cannot understand text like "Rawah" or "Comanche Peak"
# We convert text categories into numbers using LabelEncoder

# Create models folder if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Encode Wilderness_Area
le_wilderness = LabelEncoder()
df_eng['Wilderness_Area'] = le_wilderness.fit_transform(
    df_eng['Wilderness_Area']
)

# Save the encoder — we need it later for the Streamlit app
joblib.dump(le_wilderness, '../models/le_wilderness.pkl')

print(" Wilderness_Area encoded!")
print(f"Encoded values: {df_eng['Wilderness_Area'].unique()}")
print(f"Original labels: {le_wilderness.classes_}")

 Wilderness_Area encoded!
Encoded values: [0 2 3 1]
Original labels: [1 2 3 4]


# Encode Soil_Type (Label Encoding)

In [17]:
# Same process for Soil_Type

le_soil = LabelEncoder()
df_eng['Soil_Type'] = le_soil.fit_transform(
    df_eng['Soil_Type']
)

# Save the encoder
joblib.dump(le_soil, '../models/le_soil.pkl')

print(" Soil_Type encoded!")
print(f"Encoded values: {sorted(df_eng['Soil_Type'].unique())}")
print(f"Original labels: {le_soil.classes_}")

 Soil_Type encoded!
Encoded values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]
Original labels: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 16 17 18 19 20 21 22 23 24 25
 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40]


# Encode Target Variable (Cover_Type)

In [20]:
# We also need to encode the target column
# ML models need numbers, not text like "Spruce/Fir"

le_target = LabelEncoder()
df_eng['Cover_Type'] = le_target.fit_transform(
    df_eng['Cover_Type']
)

# Save this encoder — VERY important for Streamlit app
# We need it to convert predicted numbers back to forest names
joblib.dump(le_target, '../models/le_target.pkl')

print(" Cover_Type encoded!")
print(f"Encoded values: {sorted(df_eng['Cover_Type'].unique())}")
print(f"\nMapping (number → forest type):")
for i, name in enumerate(le_target.classes_):
    print(f"  {i} → {name}")

 Cover_Type encoded!
Encoded values: [0, 1, 2, 3, 4, 5, 6]

Mapping (number → forest type):
  0 → Aspen
  1 → Cottonwood/Willow
  2 → Douglas-fir
  3 → Krummholz
  4 → Lodgepole Pine
  5 → Ponderosa Pine
  6 → Spruce/Fir


#  Check Final Dataset

In [23]:
# Verify everything looks correct

print("=== FINAL DATASET AFTER FEATURE ENGINEERING ===\n")
print(f"Shape: {df_eng.shape}")
print(f"\nColumn names:\n{df_eng.columns.tolist()}")
print(f"\nData types:\n{df_eng.dtypes}")
print(f"\nAny missing values: {df_eng.isnull().sum().sum()}")
df_eng.head()

=== FINAL DATASET AFTER FEATURE ENGINEERING ===

Shape: (145890, 17)

Column names:
['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points', 'Cover_Type', 'Wilderness_Area', 'Soil_Type', 'Mean_Distance_To_Hydrology', 'Mean_Hillshade', 'Elevation_To_Roadway_Ratio', 'Hillshade_Diff']

Data types:
Elevation                             float64
Aspect                                  int64
Slope                                   int64
Horizontal_Distance_To_Hydrology      float64
Vertical_Distance_To_Hydrology          int64
Horizontal_Distance_To_Roadways         int64
Hillshade_9am                         float64
Hillshade_Noon                          int64
Hillshade_3pm                           int64
Horizontal_Distance_To_Fire_Points      int64
Cover_Type                              int32
Wilderness_Area            

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Cover_Type,Wilderness_Area,Soil_Type,Mean_Distance_To_Hydrology,Mean_Hillshade,Elevation_To_Roadway_Ratio,Hillshade_Diff
0,2596.0,51,3,258.0,0,510,221.0,232,148,6279,0,0,27,129.0,200.333333,5.080235,73.0
1,2590.0,56,2,212.0,-6,390,220.0,235,151,6225,0,0,27,103.0,202.000000,6.624041,69.0
2,2804.0,139,9,268.0,65,3180,234.0,238,135,6121,4,0,11,166.5,202.333333,0.881484,99.0
3,2785.0,155,18,242.0,117,3090,238.0,238,122,6211,4,0,28,179.5,199.333333,0.901003,116.0
4,2595.0,45,2,153.0,-1,391,220.0,234,150,6172,0,0,27,76.0,201.333333,6.619898,70.0


# Correlation of New Features with Target

In [26]:
# Check how well our NEW features correlate with Cover_Type
# Higher absolute value = more useful feature

print("=== CORRELATION OF ALL FEATURES WITH TARGET ===\n")

correlations = df_eng.corr()['Cover_Type'].drop('Cover_Type')
correlations = correlations.abs().sort_values(ascending=False)

for col, val in correlations.items():
    print(f"{col}: {val:.4f}")

=== CORRELATION OF ALL FEATURES WITH TARGET ===

Elevation: 0.4073
Wilderness_Area: 0.3747
Horizontal_Distance_To_Roadways: 0.2692
Horizontal_Distance_To_Fire_Points: 0.2045
Soil_Type: 0.1898
Slope: 0.1255
Horizontal_Distance_To_Hydrology: 0.0943
Mean_Distance_To_Hydrology: 0.0767
Hillshade_3pm: 0.0511
Vertical_Distance_To_Hydrology: 0.0493
Elevation_To_Roadway_Ratio: 0.0490
Hillshade_Diff: 0.0439
Mean_Hillshade: 0.0412
Aspect: 0.0397
Hillshade_9am: 0.0276
Hillshade_Noon: 0.0230


# Save Final Engineered Dataset

In [29]:
# Save the fully engineered dataset for Notebook 5

df_eng.to_csv('../data/forest_cover_engineered.csv', index=False)

print(" Engineered dataset saved as 'forest_cover_engineered.csv'")
print(f"Final shape: {df_eng.shape}")
print(f"\nFiles saved in models/ folder:")
print("- le_wilderness.pkl")
print("- le_soil.pkl") 
print("- le_target.pkl")

 Engineered dataset saved as 'forest_cover_engineered.csv'
Final shape: (145890, 17)

Files saved in models/ folder:
- le_wilderness.pkl
- le_soil.pkl
- le_target.pkl


## 📝 Summary from Notebook 4

- Loaded cleaned dataset: 145,891 rows × 13 columns
- Created 4 new features:
    1. Mean_Distance_To_Hydrology
    2. Mean_Hillshade
    3. Elevation_To_Roadway_Ratio
    4. Hillshade_Diff
- Encoded Wilderness_Area using LabelEncoder → saved as le_wilderness.pkl
- Encoded Soil_Type using LabelEncoder → saved as le_soil.pkl
- Encoded Cover_Type (target) using LabelEncoder → saved as le_target.pkl
- Top 3 features correlated with target:
    1. Elevation (0.4073)
    2. Wilderness_Area (0.3747)
    3. Horizontal_Distance_To_Roadways (0.2692)
- Weakest features: Hillshade_Noon (0.023), Hillshade_9am (0.028)
- Final dataset shape: 145,891 rows × 17 columns
- Saved engineered file: forest_cover_engineered.csv
- Next step: Notebook 5 - Model Building